In [21]:
import os, shutil, glob, re
from tqdm import tqdm
import numpy as np
import pandas as pd

In [2]:
base = r"D:\JHAhn\8641_conductivity\DTI_ALPS"
dst_root = os.path.join(base, "CTI_ALPS")

def ensure_tree(dst_dir: str):
    """dst_dir/Results/{CTI,Stats} 생성"""
    os.makedirs(os.path.join(dst_dir, "Results", "CTI"), exist_ok=True)
    os.makedirs(os.path.join(dst_dir, "Results", "Stats"), exist_ok=True)

def list_dirs(src: str):
    """src 하위의 1-depth 폴더 목록만"""
    try:
        for name in os.listdir(src):
            p = os.path.join(src, name)
            if os.path.isdir(p):
                yield name
    except FileNotFoundError:
        print(f"[WARN] Not found: {src}")

def process_block(src: str, exclude=None, note=""):
    """src의 폴더들(필터 적용)의 동명 폴더를 dst_root에 만들고 하위 트리 구성"""
    if exclude is None:
        exclude = lambda n: False
    created = 0
    for name in list_dirs(src):
        if exclude(name):
            continue
        dst_dir = os.path.join(dst_root, name)
        ensure_tree(dst_dir)
        created += 1
    print(f"[OK] {note} from {src} → {dst_root}  (folders processed: {created})")

In [12]:
# [1]
process_block(
    src=os.path.join(base, r"1_8641_CN_MCI"),
    exclude=lambda n: (n == "Bin") or n.startswith("x_"),
    note="[1]"
)

# [2]
process_block(
    src=os.path.join(base, r"2_Dementia", r"DKI"),
    exclude=lambda n: (n == "Bin"),
    note="[2]"
)

# [3]
process_block(
    src=os.path.join(base, r"2_Dementia", r"EPT2019", r"prep"),
    exclude=None,
    note="[3]"
)

# [4]
process_block(
    src=os.path.join(base, r"2_Dementia", r"EPT2020"),
    exclude=lambda n: n.startswith("xEPI-"),
    note="[4]"
)

# [5]
process_block(
    src=os.path.join(base, r"2_Dementia", r"EPT2021"),
    exclude=None,
    note="[5]"
)

# [6]
process_block(
    src=os.path.join(base, r"2_Dementia", r"EPT2022"),
    exclude=None,
    note="[6]"
)

# [7]
process_block(
    src=os.path.join(base, r"2_Dementia", r"EPT2023"),
    exclude=lambda n: n.startswith("xEPI-"),
    note="[7]"
)

# [8]
process_block(
    src=os.path.join(base, r"2_Dementia", r"EPT2024"),
    exclude=None,
    note="[8]"
)

# [9]
process_block(
    src=os.path.join(base, r"8641_DKI"),
    exclude=None,
    note="[9]"
)

[OK] [1] from D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 58)
[OK] [2] from D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\DKI → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 55)
[OK] [3] from D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2019\prep → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 70)
[OK] [4] from D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2020 → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 30)
[OK] [5] from D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2021 → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 2)
[OK] [6] from D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2022 → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 74)
[OK] [7] from D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2023 → D:\JHAhn\8641_conductivity\DTI_ALPS\CTI_ALPS  (folders processed: 22)
[OK] [8] from D:\J

In [13]:
cti_root    = os.path.join(base, "CTI_ALPS")
tensors_root= r"D:\JHAhn\8641_conductivity\8641_CDT_tensors"

MOVE_FILES = False  # True로 바꾸면 '복사' 대신 '이동'

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def unique_rename(path, new_name):
    parent = os.path.dirname(path)
    cand   = os.path.join(parent, new_name)
    if not os.path.exists(cand):
        os.rename(path, cand);  return cand
    i = 1
    while True:
        cand_i = os.path.join(parent, f"{new_name}_{i}")
        if not os.path.exists(cand_i):
            os.rename(path, cand_i);  return cand_i
        i += 1

def copy_or_move(src, dst):
    if MOVE_FILES:
        shutil.move(src, dst)
    else:
        shutil.copy2(src, dst)

def process_one(name):
    dst_folder = os.path.join(cti_root, name)
    if not os.path.isdir(dst_folder):
        return

    src_folder = os.path.join(tensors_root, name)
    nii_files  = glob.glob(os.path.join(src_folder, "*.nii")) if os.path.isdir(src_folder) else []

    if nii_files:
        cti_dir = os.path.join(dst_folder, "Results", "CTI")
        ensure_dir(cti_dir)
        for f in tqdm(nii_files, desc=f"[{name}] copying", unit="file"):
            try:
                copy_or_move(f, os.path.join(cti_dir, os.path.basename(f)))
            except Exception as e:
                tqdm.write(f"[ERR] {name}: {os.path.basename(f)} -> {e}")
        tqdm.write(f"[OK] {name}: {len(nii_files)} file(s) {'moved' if MOVE_FILES else 'copied'}")
    else:
        if not name.startswith("x_"):
            new_path = unique_rename(dst_folder, "x_" + name)
            tqdm.write(f"[REN] {name} -> {os.path.basename(new_path)} (no matching .nii)")
        else:
            tqdm.write(f"[SKIP] {name}: already prefixed x_, no matching .nii")

def main():
    if not os.path.isdir(cti_root):
        print(f"[WARN] Not found: {cti_root}");  return
    names = [d for d in os.listdir(cti_root)
             if os.path.isdir(os.path.join(cti_root, d))]
    for name in tqdm(sorted(names), desc="Processing folders", unit="folder"):
        process_one(name)

if __name__ == "__main__":
    main()

Processing folders:   0%|                                                                  | 0/339 [00:00<?, ?folder/s]

[OK] ADEPT01_YSJ_NODDI24: 1 file(s) copied



Processing folders:   0%|                                                                  | 0/339 [00:00<?, ?folder/s]

[OK] ADEPT02_LKY_NODDI24: 1 file(s) copied



Processing folders:   1%|▌                                                         | 3/339 [00:00<00:17, 19.70folder/s]

...

[OK] MWI_CN82_LMR_NODDI32: 1 file(s) copied



Processing folders: 100%|███████████████████████████████████████████████████████▊| 338/339 [11:19<00:02,  2.45s/folder]

[OK] MWI_CN83_LYS_NODDI32: 1 file(s) copied



Processing folders: 100%|████████████████████████████████████████████████████████| 339/339 [11:21<00:00,  2.01s/folder]

[OK] MWI_CN84_LHS_NODDI32: 1 file(s) copied


In [3]:
cti_root = os.path.join(base, "CTI_ALPS")

# === 대상 경로 블록 정의 ===
blocks = [
    # [1]
    (os.path.join(base, r"1_8641_CN_MCI"),
     lambda n: (n == "Bin") or n.startswith("x_"), "[1]"),
    # [2]
    (os.path.join(base, r"2_Dementia", r"DKI"),
     lambda n: (n == "Bin"), "[2]"),
    # [3]
    (os.path.join(base, r"2_Dementia", r"EPT2019", r"prep"),
     None, "[3]"),
    # [4]
    (os.path.join(base, r"2_Dementia", r"EPT2020"),
     lambda n: n.startswith("xEPI-"), "[4]"),
    # [5]
    (os.path.join(base, r"2_Dementia", r"EPT2021"),
     None, "[5]"),
    # [6]
    (os.path.join(base, r"2_Dementia", r"EPT2022"),
     None, "[6]"),
    # [7]
    (os.path.join(base, r"2_Dementia", r"EPT2023"),
     lambda n: n.startswith("xEPI-"), "[7]"),
    # [8]
    (os.path.join(base, r"2_Dementia", r"EPT2024"),
     None, "[8]"),
    # [9]
    (os.path.join(base, r"8641_DKI"),
     None, "[9]"),
]

In [ ]:
OVERWRITE = True   # True면 같은 파일명은 덮어씀, False면 이미 있으면 건너뜀
DRY_RUN   = False  # True면 실제 복사/생성 안 하고 로그만

def list_dirs(path):
    try:
        for n in os.listdir(path):
            p = os.path.join(path, n)
            if os.path.isdir(p):
                yield n
    except FileNotFoundError:
        return

def find_match(name):
    """블록 우선순위([1]→[9])대로 동명 폴더를 찾아 경로 반환(필터 제외 적용). 없으면 None."""
    for root, exclude, _note in blocks:
        if not os.path.isdir(root):
            continue
        if exclude and exclude(name):
            continue
        cand = os.path.join(root, name)
        if os.path.isdir(cand):
            return cand, _note
    return None, None

def ensure_dir(p):
    if DRY_RUN: return
    os.makedirs(p, exist_ok=True)

def copytree_merge(src_dir, dst_dir, pbar=None):
    """src_dir 내용을 dst_dir로 병합 복사."""
    for root, dirs, files in os.walk(src_dir):
        rel = os.path.relpath(root, src_dir)
        target_root = dst_dir if rel == "." else os.path.join(dst_dir, rel)
        if not DRY_RUN:
            os.makedirs(target_root, exist_ok=True)
        for f in files:
            src_f = os.path.join(root, f)
            dst_f = os.path.join(target_root, f)
            if os.path.exists(dst_f) and not OVERWRITE:
                if pbar: pbar.write(f"[SKIP] {os.path.relpath(dst_f, dst_dir)} (exists)")
                continue
            if not DRY_RUN:
                shutil.copy2(src_f, dst_f)
            if pbar: pbar.update(1)

def count_files(src_dir):
    total = 0
    for _, _, files in os.walk(src_dir):
        total += len(files)
    return total

def main():
    if not os.path.isdir(cti_root):
        print(f"[ERR] CTI root not found: {cti_root}")
        return

    cti_names = [d for d in os.listdir(cti_root)
                 if os.path.isdir(os.path.join(cti_root, d))]
    cti_names.sort()

    outer = tqdm(cti_names, desc="CTI_ALPS folders", unit="folder")
    for name in outer:
        if name.startswith("x_"):
            outer.write(f"[SKIP] {name}: prefixed x_")
            continue

        cti_src = os.path.join(cti_root, name, "Results", "CTI")
        if not os.path.isdir(cti_src):
            outer.write(f"[MISS] {name}: CTI source missing → {cti_src}")
            continue

        dest_person, note = find_match(name)
        if not dest_person:
            outer.write(f"[NOMATCH] {name}: no matching person in target blocks")
            continue

        results_dir = os.path.join(dest_person, "Results")
        dest_cti    = os.path.join(results_dir, "CTI")
        ensure_dir(results_dir)
        ensure_dir(dest_cti)

        # 파일 개수 세고 tqdm 바
        n_files = count_files(cti_src)
        if n_files == 0:
            outer.write(f"[EMPTY] {name}: CTI has no files")
            continue

        desc = f"[{note} {name}] → Results\\CTI"
        with tqdm(total=n_files, desc=desc, unit="file", leave=False) as pbar:
            copytree_merge(cti_src, dest_cti, pbar=pbar)

        outer.write(f"[OK] {name}: copied {n_files} file(s) to {dest_cti}")

if __name__ == "__main__":
    main()

CTI_ALPS folders:   0%|▏                                                           | 1/262 [00:02<12:39,  2.91s/folder]

[OK] ADEPT01_YSJ_NODDI24: copied 4 file(s) to D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2019\prep\ADEPT01_YSJ_NODDI24\Results\CTI



CTI_ALPS folders:   1%|▍                                                           | 2/262 [00:04<08:37,  1.99s/folder]

[OK] ADEPT02_LKY_NODDI24: copied 1 file(s) to D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2019\prep\ADEPT02_LKY_NODDI24\Results\CTI



CTI_ALPS folders:   1%|▋                                                           | 3/262 [00:05<07:45,  1.80s/folder]

...

[OK] MWI_CN82_LMR_NODDI32: copied 1 file(s) to D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN82_LMR_NODDI32\Results\CTI



CTI_ALPS folders: 100%|█████████████████████████████████████████████████████████▊| 261/262 [11:26<00:02,  2.20s/folder]

[OK] MWI_CN83_LYS_NODDI32: copied 1 file(s) to D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN83_LYS_NODDI32\Results\CTI



CTI_ALPS folders: 100%|██████████████████████████████████████████████████████████| 262/262 [11:30<00:00,  2.63s/folder]

[OK] MWI_CN84_LHS_NODDI32: copied 1 file(s) to D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN84_LHS_NODDI32\Results\CTI


In [ ]:
## CTI-ALPS data merging

In [5]:
LT_COLS = ["LT_Proj_Dxx","LT_Proj_Dyy","LT_Proj_Dzz",
           "LT_Assoc_Dxx","LT_Assoc_Dyy","LT_Assoc_Dzz",
           "LT_CTI-ALPS index"]
RT_COLS = ["RT_Proj_Dxx","RT_Proj_Dyy","RT_Proj_Dzz",
           "RT_Assoc_Dxx","RT_Assoc_Dyy","RT_Assoc_Dzz",
           "RT_CTI-ALPS index"]
FINAL_COLS = ["PDI"] + LT_COLS + RT_COLS

def get_blocks(base):
    """9개 대상 경로(필터 포함) 구성"""
    return [
        (os.path.join(base, r"1_8641_CN_MCI"),                   lambda n: (n == "Bin") or n.startswith("x_")),
        (os.path.join(base, r"2_Dementia", r"DKI"),              lambda n: (n == "Bin")),
        (os.path.join(base, r"2_Dementia", r"EPT2019", r"prep"), None),
        (os.path.join(base, r"2_Dementia", r"EPT2020"),          lambda n: n.startswith("xEPI-")),
        (os.path.join(base, r"2_Dementia", r"EPT2021"),          None),
        (os.path.join(base, r"2_Dementia", r"EPT2022"),          None),
        (os.path.join(base, r"2_Dementia", r"EPT2023"),          lambda n: n.startswith("xEPI-")),
        (os.path.join(base, r"2_Dementia", r"EPT2024"),          None),
        (os.path.join(base, r"8641_DKI"),                        None),
    ]

def _norm_cell(x):
    if pd.isna(x): return None
    s = str(x).strip().strip('"').strip()
    v = pd.to_numeric(s, errors="coerce")
    return v if v is not None and not pd.isna(v) else s

def read_stat_file(path):
    """한 행 파일 읽어 (pdi:str, 7개 float 리스트) 반환. 실패 시 None"""
    try:
        ext = os.path.splitext(path)[1].lower()
        if ext in (".xlsx", ".xls"):
            df = pd.read_excel(path, header=None)
        else:
            df = pd.read_csv(path, header=None, sep=None, engine="python")
        if df.empty: return None
        row = [_norm_cell(x) for x in df.iloc[0].tolist()]
        if len(row) < 8: return None
        pdi  = str(row[0]).strip().strip('"').strip()
        vals = [pd.to_numeric(v, errors="coerce") for v in row[1:8]]
        if any(pd.isna(v) for v in vals): return None
        return pdi, vals
    except Exception:
        return None

def find_person_roots(blocks):
    """블록 우선순위대로 사람 폴더 매핑(name -> abs_path)"""
    seen = {}
    for root, exclude in blocks:
        if not os.path.isdir(root): continue
        try:
            for name in os.listdir(root):
                p = os.path.join(root, name)
                if not os.path.isdir(p): continue
                if exclude and exclude(name): continue
                if name not in seen:
                    seen[name] = p
        except PermissionError:
            continue
    return seen

def find_stat_paths(person_dir):
    """person_dir/Results/Stats 에서 LH/RH 파일 경로"""
    stats_dir = os.path.join(person_dir, "Results", "Stats")
    if not os.path.isdir(stats_dir): return None, None
    c_lh = glob.glob(os.path.join(stats_dir, "Auto_CTI_ALPS_index_LH.*"))
    c_rh = glob.glob(os.path.join(stats_dir, "Auto_CTI_ALPS_index_RH.*"))
    return (c_lh[0] if c_lh else None), (c_rh[0] if c_rh else None)

def make_row(person_dir):
    """LH/RH 둘 다 있으면 1행 dict, 아니면 None"""
    lh_path, rh_path = find_stat_paths(person_dir)
    if not lh_path or not rh_path: return None
    lh = read_stat_file(lh_path)
    rh = read_stat_file(rh_path)
    if (lh is None) or (rh is None): return None

    pdi_lh, vals_lh = lh
    _pdi_rh, vals_rh = rh
    row = {"PDI": pdi_lh}
    keys = ["Proj_Dxx","Proj_Dyy","Proj_Dzz","Assoc_Dxx","Assoc_Dyy","Assoc_Dzz","CTI-ALPS index"]
    for k, v in zip(keys, vals_lh): row["LT_"+k] = float(v)
    for k, v in zip(keys, vals_rh): row["RT_"+k] = float(v)
    return row

def merge_cti_alps(base, show_progress=True):
    """
    반환: (df, info)
      - df: 최종 병합 DataFrame (PDI 1열 + LT/RT 7+7열)
      - info: {'total':N, 'written':M, 'skipped':K}
    """
    blocks_raw = get_blocks(base)
    # (root, exclude, note) → (root, exclude)만 사용
    blocks = [(r, ex) for (r, ex, *_) in [(b[0], b[1], None) if len(b)==2 else b for b in blocks_raw]]

    people = find_person_roots(blocks)
    names = sorted(people.keys())
    rows, skipped = [], 0

    iterator = tqdm(names, desc="Merging CTI-ALPS", unit="person") if show_progress else names
    for name in iterator:
        row = make_row(people[name])
        if row is None:
            skipped += 1
            if show_progress: tqdm.write(f"[SKIP] {name}: LH/RH 없음 또는 읽기 실패")
            continue
        rows.append(row)

    if not rows:
        empty = pd.DataFrame(columns=FINAL_COLS)
        return empty, {"total": len(names), "written": 0, "skipped": skipped}

    df = pd.DataFrame(rows)
    for c in FINAL_COLS:
        if c not in df.columns: df[c] = pd.NA
    df = df[FINAL_COLS]
    return df, {"total": len(names), "written": len(df), "skipped": skipped}

df, info = merge_cti_alps(base, show_progress=True)
print(info)
df

Merging CTI-ALPS:  28%|████████████████▍                                         | 96/340 [00:00<00:00, 474.71person/s]

[SKIP] Bin: LH/RH 없음 또는 읽기 실패
[SKIP] EPT2022AD04_KSS_NODDI32: LH/RH 없음 또는 읽기 실패
[SKIP] EPT2022AD28_GMS_NODDI32v: LH/RH 없음 또는 읽기 실패


Merging CTI-ALPS:  55%|███████████████████████████████▎                         | 187/340 [00:00<00:00, 403.77person/s]

[SKIP] HFC2020AD01_KYM_NODDI32: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD03_LHY_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD04_HKS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD05_KSM_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD06_KKO_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD07_HJY_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD08_NHN_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD09_KID_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD10_KSY_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD11_KJB_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD12_LGH_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD13_KYI_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD14_PGJ_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD15_YOH_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD16_OGY_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD17_KJS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD18_WJC_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD19_KDH_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD20_YSJ_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD21_CYS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD22_LUB_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD23_OSD_DKI: LH

Merging CTI-ALPS:  80%|█████████████████████████████████████████████▊           | 273/340 [00:00<00:00, 366.04person/s]

[SKIP] HFC2021AD48_KJM_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD49_LJS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD50_DGY_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD51_NEH_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD52_KSS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD53_LSD_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD54_KGI_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD55_LGE_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD56_SEY_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] HFC2021AD57_PGJ_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MREPT2020AD09_PJJ_NODDI32: LH/RH 없음 또는 읽기 실패
[SKIP] MREPT2020AD16_CKL_NODDI32: LH/RH 없음 또는 읽기 실패
[SKIP] MREPT2020AD17_LSH_NODDI32: LH/RH 없음 또는 읽기 실패
[SKIP] MREPT2020AD21_CBS_NODDI32: LH/RH 없음 또는 읽기 실패


Merging CTI-ALPS: 100%|█████████████████████████████████████████████████████████| 340/340 [00:00<00:00, 385.78person/s]

[SKIP] MWI_CN42_KKD_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN43_LSL_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN44_OKS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN45_KTJ_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN46_KCS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN47_YKS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN48_WHJ_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN49_CBW_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN50_PMJ_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN51_POS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN52_SJS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN53_YMS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN54_KHS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN55_JMS_DKI: LH/RH 없음 또는 읽기 실패
[SKIP] MWI_CN56_KYO_DKI: LH/RH 없음 또는 읽기 실패
{'total': 340, 'written': 262, 'skipped': 78}


,PDI,LT_Proj_Dxx,LT_Proj_Dyy,LT_Proj_Dzz,LT_Assoc_Dxx,LT_Assoc_Dyy,LT_Assoc_Dzz,LT_CTI-ALPS index,RT_Proj_Dxx,RT_Proj_Dyy,RT_Proj_Dzz,RT_Assoc_Dxx,RT_Assoc_Dyy,RT_Assoc_Dzz,RT_CTI-ALPS index
0,ADEPT01_YSJ_NODDI24,0.154640,0.136547,0.372179,0.286533,0.394715,0.262895,1.104470,0.421284,0.419805,0.658573,0.393475,0.464674,0.372838,1.027901
1,ADEPT02_LKY_NODDI24,0.230889,0.213943,0.426038,0.247660,0.395434,0.240741,1.052484,0.206692,0.183261,0.478556,0.305591,0.433569,0.313158,1.031956
2,ADEPT03_YKJ_NODDI24,0.200499,0.209197,0.430680,0.340928,0.385080,0.309460,1.043901,0.370849,0.376692,0.702424,0.350790,0.394135,0.352442,0.989719
3,ADEPT04_JWS_NODDI24,0.331312,0.331995,0.598961,0.426246,0.447030,0.377781,1.067319,0.410472,0.408417,0.720397,0.390904,0.460661,0.391659,1.001624
4,ADEPT05_CBK_NODDI24,0.361086,0.332316,0.500665,0.263974,0.370518,0.238849,1.094361,0.440325,0.428100,0.646132,0.350006,0.462110,0.330923,1.041247
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,MWI_CN80_LSR_NODDI32,0.120455,0.116088,0.334192,0.272016,0.299302,0.205978,1.218601,0.095568,0.095638,0.362974,0.188456,0.319088,0.206177,0.941056
258,MWI_CN81_CSH_NODDI32,0.099054,0.085795,0.268894,0.225133,0.252498,0.184627,1.198815,0.128332,0.117998,0.345852,0.233705,0.350583,0.250176,0.983328
259,MWI_CN82_LMR_NODDI32,0.180362,0.149926,0.326837,0.200367,0.295091,0.177061,1.164355,0.202879,0.205939,0.485830,0.220892,0.379424,0.203464,1.035095
260,MWI_CN83_LYS_NODDI32,0.072631,0.057072,0.236033,0.205523,0.305449,0.174870,1.199239,0.099302,0.093738,0.271875,0.156811,0.200440,0.142957,1.082038


In [6]:
# Save to file
def save_cti_alps(df, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with pd.ExcelWriter(out_path, engine="openpyxl") as w:
        df.to_excel(w, sheet_name="CTI_ALPS_index", index=False)
    return out_path

save_cti_alps(df, os.path.join(base, "CTI_ALPS", "CTI_ALPS-index_Results.xlsx"))

'D:\\JHAhn\\8641_conductivity\\DTI_ALPS\\CTI_ALPS\\CTI_ALPS-index_Results.xlsx'

In [17]:
main = pd.read_excel(r"D:\JHAhn\files\Auto_ALPS_index_19-24_main.xlsx")
main["PID"] = (main["PID"].astype(str)
                          .str.strip()
                          .str.replace(r"\.0$", "", regex=True))
main["PID"] = main["PID"].mask(main["PID"].eq("") | main["PID"].str.lower().eq("nan"))
mask = main["PID"].notna()
main.loc[mask, "PID"] = main.loc[mask, "PID"].str.zfill(8)
# main = main.set_index("PID")
main

,PDI,PID,DDX,Group,Age,Sex_01,Sex,Education (Year),MMSE,CDR,...,R_LT_insulaIC-HippoVol,R_RT_insulaIC-HippoVol,R_LT_HippoHFC-HippoVol,R_RT_HippoHFC-HippoVol,R_LT_HippoEC-HippoVol,R_RT_HippoEC-HippoVol,R_LT_InsulaHFC-InsulaVol,R_RT_InsulaHFC-InsulaVol,R_LT_InsulaEC-InsulaVol,R_RT_InsulaEC-InsulaVol
0,ADEPT54_HCS_NODDI24,02459562,NL,0,82,1,F,6.0,27,0.0,...,0.305024,0.205650,2.696677,3.256284,2.444712,3.149281,1.794523,2.122095,1.687050,2.056645
1,EPT2022AD01_KDKM_NODDI32,00310282,AD,2,66,1,F,NaN,23,0.5,...,0.503716,0.338423,1.950329,2.288472,1.344138,1.474099,0.984815,1.160552,0.757258,1.013958
2,EPT2022AD02_SKM_NODDI32,00454439,AD,2,74,0,M,NaN,13,2.0,...,0.259266,0.242416,4.028479,3.885575,3.000292,3.186409,3.174278,3.362853,2.911189,3.385903
3,EPT2022AD03_KKS_NODDI32,00461257,SaMCI,1,74,1,F,12.0,27,0.5,...,0.590434,0.451437,4.347021,4.789124,3.108060,3.079740,1.929451,2.280834,1.681998,2.082624
4,EPT2022AD05_KIJD_NODDI32,00612238,AD,2,83,1,F,NaN,24,1.0,...,0.499354,0.355829,5.133861,3.923252,3.753946,2.823341,1.630049,1.702562,1.476418,1.624511
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,MWI_CN79_CYE_NODDI32,10412739,SaMCI,1,72,1,F,NaN,28,0.5,...,0.392295,0.329853,2.061915,2.584288,1.402835,1.590919,0.985638,1.137157,0.793481,1.013646
98,MWI_CN80_LSR_NODDI32,10474812,MaMCI,1,78,1,F,NaN,29,0.5,...,0.436719,0.321682,2.986758,3.788903,2.069485,2.511578,1.512010,1.565580,1.362344,1.507729
99,MWI_CN81_CSH_NODDI32,10482577,SaMCI,1,66,0,M,NaN,28,0.5,...,0.364005,0.253456,3.074656,3.210483,2.783346,2.181404,1.241762,1.372388,1.100934,1.277327
100,MWI_CN82_LMR_NODDI32,10412747,SaMCI,1,74,0,M,NaN,28,0.5,...,0.411900,0.321186,2.727216,3.716997,2.377508,2.466592,1.344976,1.582950,1.148864,1.442690


In [18]:
if "PDI" not in main.columns:
    raise KeyError("No 'PDI' col in file 'main'")
if "PDI" not in df.columns:
    raise KeyError("No 'PDI' col in file 'df'")
main["PDI"] = (main["PDI"].astype(str).str.strip().str.strip('"').str.strip())
df["PDI"]   = (df["PDI"].astype(str).str.strip().str.strip('"').str.strip())
main["PDI"] = main["PDI"].mask(main["PDI"].eq("") | main["PDI"].str.lower().eq("nan"))
df["PDI"]   = df["PDI"].mask(df["PDI"].eq("")   | df["PDI"].str.lower().eq("nan"))

dup_main = main["PDI"].duplicated(keep=False).sum()
dup_df   = df["PDI"].duplicated(keep=False).sum()
if dup_main: print(f"[WARN] Duplicated PDIs in main: {dup_main} --> Use first PDI")
if dup_df:   print(f"[WARN] Duplicated PDIs in df {dup_df} --> Use first PDI")
main = main.drop_duplicates(subset=["PDI"], keep="first")
df   = df.drop_duplicates(subset=["PDI"], keep="first")

new_cols = [c for c in df.columns if c != "PDI"]
df = df.rename(columns={c: f"CTI_{c}" for c in new_cols})

merged = main.merge(df, on="PDI", how="left")

merged

,PDI,PID,DDX,Group,Age,Sex_01,Sex,Education (Year),MMSE,CDR,...,CTI_CTI_CTI_LT_Assoc_Dyy,CTI_CTI_CTI_LT_Assoc_Dzz,CTI_CTI_CTI_LT_CTI-ALPS index,CTI_CTI_CTI_RT_Proj_Dxx,CTI_CTI_CTI_RT_Proj_Dyy,CTI_CTI_CTI_RT_Proj_Dzz,CTI_CTI_CTI_RT_Assoc_Dxx,CTI_CTI_CTI_RT_Assoc_Dyy,CTI_CTI_CTI_RT_Assoc_Dzz,CTI_CTI_CTI_RT_CTI-ALPS index
0,ADEPT54_HCS_NODDI24,02459562,NL,0,82,1,F,6.0,27,0.0,...,0.472277,0.365113,1.109783,0.520848,0.521945,0.838614,0.467179,0.576006,0.492187,0.974257
1,EPT2022AD01_KDKM_NODDI32,00310282,AD,2,66,1,F,NaN,23,0.5,...,0.183024,0.106396,1.464138,0.065613,0.067988,0.281590,0.176224,0.300066,0.182062,0.967150
2,EPT2022AD02_SKM_NODDI32,00454439,AD,2,74,0,M,NaN,13,2.0,...,0.660463,0.603218,1.030054,0.572164,0.576388,0.822231,0.644952,0.752475,0.651444,0.991272
3,EPT2022AD03_KKS_NODDI32,00461257,SaMCI,1,74,1,F,12.0,27,0.5,...,0.237381,0.136555,1.267073,0.154561,0.157407,0.506764,0.192443,0.356904,0.196123,0.981540
4,EPT2022AD05_KIJD_NODDI32,00612238,AD,2,83,1,F,NaN,24,1.0,...,0.288539,0.223676,1.173859,0.304080,0.295770,0.657524,0.360456,0.467134,0.333783,1.055569
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,MWI_CN79_CYE_NODDI32,10412739,SaMCI,1,72,1,F,NaN,28,0.5,...,0.200644,0.137980,1.401680,0.060288,0.061287,0.217723,0.152784,0.181179,0.100257,1.318971
98,MWI_CN80_LSR_NODDI32,10474812,MaMCI,1,78,1,F,NaN,29,0.5,...,0.299302,0.205978,1.218601,0.095568,0.095638,0.362974,0.188456,0.319088,0.206177,0.941056
99,MWI_CN81_CSH_NODDI32,10482577,SaMCI,1,66,0,M,NaN,28,0.5,...,0.252498,0.184627,1.198815,0.128332,0.117998,0.345852,0.233705,0.350583,0.250176,0.983328
100,MWI_CN82_LMR_NODDI32,10412747,SaMCI,1,74,0,M,NaN,28,0.5,...,0.295091,0.177061,1.164355,0.202879,0.205939,0.485830,0.220892,0.379424,0.203464,1.035095


In [20]:
merged.to_excel(r"D:\JHAhn\files\Auto_ALPS_index_19-24_main_+CTI.xlsx", index=False)

In [27]:
# for Repeated test
base = r'D:\JHAhn\CTI_repeatbility\Val'


def is_num(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not pd.isna(x)

def is_filled(v):
    if pd.isna(v):
        return False
    if isinstance(v, str) and v.strip() == "":
        return False
    return True

def to_float_safe(s):
    s = str(s).strip()
    if s == "":
        return None
    # ".123" -> "0.123", "-.123" -> "-0.123"
    if re.match(r'^-?\.\d+$', s):
        s = s.replace('.', '0.', 1)
    if re.match(r'^-?\d{1,3}(,\d{3})+(\.\d+)?$', s):
        s = s.replace(',', '')
    try:
        return float(s)
    except Exception:
        try:
            return float(s.replace(',', '.'))
        except Exception:
            return None

def read_first_row(path):
    """일반 CSV 첫 행(헤더 있는 형식) → Series. 실패/빈 파일은 None."""
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"[WARN] read_csv failed: {path} -> {e}")
        return None
    if df.empty:
        print(f"[WARN] empty CSV skipped: {path}")
        return None
    df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed')]
    df.columns = df.columns.map(lambda x: str(x).strip())
    s = df.iloc[0].copy()
    for c in s.index:
        try:
            s[c] = pd.to_numeric(s[c])
        except Exception:
            pass
    return s

def read_cti_singleline(path, side):
    """
    1-lined CTI CSV without Header:
      PDI, Dxx_proj, Dyy_proj, Dzz_proj, Dxx_assoc, Dyy_assoc, Dzz_assoc, ALPS index
    → {side}_CTI_* 표준 컬럼명으로 Series 반환.
    """
    try:
        raw = open(path, 'r', encoding='utf-8', errors='ignore').read().strip()
    except Exception as e:
        print(f"[WARN] read failed: {path} -> {e}")
        return None

    parts = [p.strip() for p in raw.split(',') if p.strip() != ""]
    if len(parts) < 8:
        parts = re.split(r'[,\t]+', raw)
        parts = [p.strip() for p in parts if p.strip() != ""]
    if len(parts) < 8:
        print(f"[WARN] CTI single-line format not matched: {path}")
        return None

    pdi_from_file = parts[0]
    nums = [to_float_safe(x) for x in parts[1:8]]
    if any(v is None for v in nums):
        print(f"[WARN] CTI single-line numeric parse failed: {path} -> {parts[1:8]}")
        return None

    proj_dxx, proj_dyy, proj_dzz, assoc_dxx, assoc_dyy, assoc_dzz, alps = nums
    ser = pd.Series({
        f"{side}_CTI_Proj_Dxx": proj_dxx,
        f"{side}_CTI_Proj_Dyy": proj_dyy,
        f"{side}_CTI_Proj_Dzz": proj_dzz,
        f"{side}_CTI_Assoc_Dxx": assoc_dxx,
        f"{side}_CTI_Assoc_Dyy": assoc_dyy,
        f"{side}_CTI_Assoc_Dzz": assoc_dzz,
        f"{side}_CTI-ALPS index": alps,
        # 참고용: 파일 안의 PDI (폴더명과 다르면 확인용)
        "PDI_from_file": pdi_from_file,
    })
    return ser

def find_file(stats_dir, filename):
    pats = glob.glob(os.path.join(stats_dir, '**', filename), recursive=True)
    return pats[0] if pats else None

# Schema targeted
dti_feats = ['Dxx_proj','Dyy_proj','Dzz_proj','Dxx_assoc','Dyy_assoc','Dzz_assoc','ALPS index']
dti_bands = ['ALL', 'b800', 'b2000']
sides     = ['LT', 'RT']
final_dti_cols = [f"{side}_{band}_{feat}" for side in sides for band in dti_bands for feat in dti_feats]

final_cti_lt = ['LT_CTI_Proj_Dxx','LT_CTI_Proj_Dyy','LT_CTI_Proj_Dzz',
                'LT_CTI_Assoc_Dxx','LT_CTI_Assoc_Dyy','LT_CTI_Assoc_Dzz','LT_CTI-ALPS index']
final_cti_rt = ['RT_CTI_Proj_Dxx','RT_CTI_Proj_Dyy','RT_CTI_Proj_Dzz',
                'RT_CTI_Assoc_Dxx','RT_CTI_Assoc_Dyy','RT_CTI_Assoc_Dzz','RT_CTI-ALPS index']

final_cols = ['PDI'] + final_dti_cols + final_cti_lt + final_cti_rt

# Mapping target names
dti_file_map = {
    ('LT','ALL'):   'Auto_ALPS_index_LH.csv',
    ('LT','b800'):  'Auto_ALPS_index_LH_maxbval800.csv',
    ('LT','b2000'): 'Auto_ALPS_index_LH_maxbval2000.csv',
    ('RT','ALL'):   'Auto_ALPS_index_RH.csv',
    ('RT','b800'):  'Auto_ALPS_index_RH_maxbval800.csv',
    ('RT','b2000'): 'Auto_ALPS_index_RH_maxbval2000.csv',
}
cti_file_map = {
    'LT': 'Auto_CTI_ALPS_index_LH.csv',
    'RT': 'Auto_CTI_ALPS_index_RH.csv',
}

# Remaning: DTI (line-by-line) → Target names
def rename_dti_series(s, side, band):
    out = {}
    for feat in dti_feats:
        candidates = [feat]
        if feat == 'ALPS index':
            candidates += ['ALPS_index','ALPS-index','ALPS','Index','ALPS  index']
        for cand in candidates:
            if cand in s.index and is_filled(s[cand]):
                out[f"{side}_{band}_{feat}"] = s[cand]
                break
    return pd.Series(out)

# Main loops
rows = []
subjects = sorted([d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))])

for pdi in tqdm(subjects, desc="Merging ALPS per subject", unit="subj", dynamic_ncols=True):
    stats_dir = os.path.join(base, pdi, 'Results', 'Stats')
    if not os.path.isdir(stats_dir):
        continue

    row = {'PDI': pdi}

    # DTI: LT/RT × ALL/b800/b2000
    for side in sides:
        for band in dti_bands:
            fp = find_file(stats_dir, dti_file_map[(side, band)])
            if not fp:
                continue
            s = read_first_row(fp)
            if s is None:
                continue
            s_renamed = rename_dti_series(s, side, band)
            for k, v in s_renamed.items():
                if is_filled(v):
                    row[k] = v

    # CTI: Parsing 1-line Without header CSV (Priority)
    for side in ['LT', 'RT']:
        fp = find_file(stats_dir, cti_file_map[side])
        if fp:
            s_cti = read_cti_singleline(fp, side)
            if s_cti is not None:
                # PDI correction test
                pf = s_cti.get('PDI_from_file')
                if isinstance(pf, str) and pf and pf != pdi:
                    print(f"[INFO] CTI PDI mismatch? folder={pdi} vs file={pf} @ {fp}")
                # Insert values
                for k, v in s_cti.items():
                    if k == 'PDI_from_file':
                        continue
                    if is_filled(v):
                        row[k] = v
                continue

        # CSV files with headers
        if fp:
            s_generic = read_first_row(fp)
            if s_generic is not None:
                def get_val(keys):
                    for k in keys:
                        if k in s_generic.index and is_filled(s_generic[k]):
                            return s_generic[k]
                    return None
                mapping = {
                    f"{side}_CTI_Proj_Dxx": get_val([f"{side}_CTI_Proj_Dxx", f"{side}_Proj_Dxx", "Proj_Dxx"]),
                    f"{side}_CTI_Proj_Dyy": get_val([f"{side}_CTI_Proj_Dyy", f"{side}_Proj_Dyy", "Proj_Dyy"]),
                    f"{side}_CTI_Proj_Dzz": get_val([f"{side}_CTI_Proj_Dzz", f"{side}_Proj_Dzz", "Proj_Dzz"]),
                    f"{side}_CTI_Assoc_Dxx": get_val([f"{side}_CTI_Assoc_Dxx", f"{side}_Assoc_Dxx", "Assoc_Dxx"]),
                    f"{side}_CTI_Assoc_Dyy": get_val([f"{side}_CTI_Assoc_Dyy", f"{side}_Assoc_Dyy", "Assoc_Dyy"]),
                    f"{side}_CTI_Assoc_Dzz": get_val([f"{side}_CTI_Assoc_Dzz", f"{side}_Assoc_Dzz", "Assoc_Dzz"]),
                    f"{side}_CTI-ALPS index": get_val([f"{side}_CTI-ALPS index", f"{side}_CTI_ALPS index", f"{side}_ALPS index", "CTI-ALPS index", "ALPS index"]),
                }
                for k, v in mapping.items():
                    if is_filled(v):
                        row[k] = v

    rows.append(row)

# Final DataFrame --> matching Column ORDERs
dti_feats = ['Dxx_proj','Dyy_proj','Dzz_proj','Dxx_assoc','Dyy_assoc','Dzz_assoc','ALPS index']
dti_bands = ['ALL', 'b800', 'b2000']
sides     = ['LT', 'RT']
final_dti_cols = [f"{side}_{band}_{feat}" for side in sides for band in dti_bands for feat in dti_feats]
final_cti_lt = ['LT_CTI_Proj_Dxx','LT_CTI_Proj_Dyy','LT_CTI_Proj_Dzz',
                'LT_CTI_Assoc_Dxx','LT_CTI_Assoc_Dyy','LT_CTI_Assoc_Dzz','LT_CTI-ALPS index']
final_cti_rt = ['RT_CTI_Proj_Dxx','RT_CTI_Proj_Dyy','RT_CTI_Proj_Dzz',
                'RT_CTI_Assoc_Dxx','RT_CTI_Assoc_Dyy','RT_CTI_Assoc_Dzz','RT_CTI-ALPS index']
final_cols = ['PDI'] + final_dti_cols + final_cti_lt + final_cti_rt

df_all = pd.DataFrame(rows).reindex(columns=final_cols)

# Numeric cols
for c in df_all.columns:
    if c != 'PDI':
        df_all[c] = pd.to_numeric(df_all[c], errors='coerce')

print("Shape:", df_all.shape)

df_all

Merging ALPS per subject:  69%|█████████████████████████████████████▏                | 11/16 [00:00<00:00, 97.38subj/s]

[WARN] CTI single-line format not matched: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Post\Results\Stats\Auto_CTI_ALPS_index_LH.csv
[WARN] empty CSV skipped: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Post\Results\Stats\Auto_CTI_ALPS_index_LH.csv
[WARN] CTI single-line format not matched: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Post\Results\Stats\Auto_CTI_ALPS_index_RH.csv
[WARN] empty CSV skipped: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Post\Results\Stats\Auto_CTI_ALPS_index_RH.csv
[WARN] CTI single-line format not matched: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Pre\Results\Stats\Auto_CTI_ALPS_index_LH.csv
[WARN] empty CSV skipped: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Pre\Results\Stats\Auto_CTI_ALPS_index_LH.csv
[WARN] CTI single-line format not matched: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Pre\Results\Stats\Auto_CTI_ALPS_index_RH.csv
[WARN] empty CSV skipped: D:\JHAhn\CTI_repeatbility\Val\HYPEPT02_Pre\Results\Stats\Auto_CTI_ALPS_index_RH.csv
[WARN] CTI single-line format not matched: D:\JH

Merging ALPS per subject: 100%|██████████████████████████████████████████████████████| 16/16 [00:00<00:00, 87.73subj/s]

Shape: (16, 57)


,PDI,LT_ALL_Dxx_proj,LT_ALL_Dyy_proj,LT_ALL_Dzz_proj,LT_ALL_Dxx_assoc,LT_ALL_Dyy_assoc,LT_ALL_Dzz_assoc,LT_ALL_ALPS index,LT_b800_Dxx_proj,LT_b800_Dyy_proj,...,LT_CTI_Assoc_Dyy,LT_CTI_Assoc_Dzz,LT_CTI-ALPS index,RT_CTI_Proj_Dxx,RT_CTI_Proj_Dyy,RT_CTI_Proj_Dzz,RT_CTI_Assoc_Dxx,RT_CTI_Assoc_Dyy,RT_CTI_Assoc_Dzz,RT_CTI-ALPS index
0,HYPEPT02_Post,0.670685,0.668682,1.103914,0.810171,1.064792,0.603963,1.163605,0.666788,0.668147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HYPEPT02_Pre,0.693707,0.555415,1.129927,0.804811,1.077580,0.484312,1.441261,0.693436,0.553563,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HYPEPT08_Post,0.724638,0.787779,1.101371,0.794060,1.100974,0.513703,1.166899,0.721739,0.784170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HYPEPT08_Pre,0.659906,0.684717,1.069681,0.716962,1.164287,0.414090,1.253057,0.657055,0.678516,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HYPEPT14_Post,0.599062,0.601874,1.067199,0.784866,1.015246,0.544233,1.207503,0.597958,0.603099,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,HYPEPT14_Pre,0.720254,0.682554,1.194912,0.980216,1.113577,0.600612,1.325214,0.720476,0.678191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,HYPEPT18_Post,0.793160,0.602836,1.110248,0.853183,1.157009,0.422863,1.605094,0.792804,0.598347,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,HYPEPT18_Pre,0.715555,0.443824,0.989053,0.739391,1.026480,0.384979,1.755479,0.715804,0.439802,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,HYPEPT19_Post,0.678935,0.531603,1.058283,0.798490,1.031517,0.434660,1.529009,0.676652,0.529352,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,HYPEPT19_Pre,0.668866,0.515009,0.945164,0.726756,0.942204,0.308454,1.694821,0.666833,0.513865,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# os.makedirs(out_dir, exist_ok=True)
df_all.to_excel(os.path.join(base, "merged-RepeatedPTs_ALPS_DTI+CTI.xlsx"), index=False)